In [1]:
!pip -q install pymupdf pdfplumber pandas openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 106.6 MB/s eta 0:00:00


In [2]:
import os

# Your paths
IN_DIR = "/content/input_format"
OUT_DIR = "/content/output_json"

# Make sure folders exist
os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Detect first PDF inside input folder
pdf_files = [f for f in os.listdir(IN_DIR) if f.lower().endswith(".pdf")]
if not pdf_files:
    raise FileNotFoundError(f"No PDF found in {IN_DIR}. Please upload your USB PD PDF there.")
else:
    pdf_path = os.path.join(IN_DIR, pdf_files[0])
    print("PDF found ✅:", pdf_path)
    print("Output folder:", OUT_DIR)



PDF found ✅: /content/input_format/USB_PD_R3_2 V1.1 2024-10.pdf
Output folder: /content/output_json


In [3]:
import fitz  # PyMuPDF

# Open PDF
doc = fitz.open(pdf_path)

# Extract TOC (list of [level, title, page])
raw_toc = doc.get_toc()
print("TOC rows found:", len(raw_toc))

# Show first 15 TOC entries for a quick check
for row in raw_toc[:15]:
    print(row)

doc.close()


TOC rows found: 1360
[1, 'Revision History', 0]
[1, 'LIMITED COPYRIGHT LICENSE', 0]
[1, 'INTELLECTUAL PROPERTY DISCLAIMER', 0]
[1, 'Editors', 0]
[1, 'Contributors', 0]
[1, 'Table Of Contents', 0]
[1, 'List of Figures', 0]
[1, 'List of Tables', 0]
[1, '1 Introduction', 0]
[2, '1.1 Overview', 0]
[2, '1.2 Purpose', 0]
[3, '1.2.1 Scope', 0]
[2, '1.3 Section Overview', 0]
[2, '1.4 Conventions', 0]
[3, '1.4.1 Precedence', 0]


In [4]:
import json, re

def normalize_toc(raw_toc, doc_title):
    entries = []
    counters = [0] * 10  # for auto-numbering if no section numbers

    for level, title, page in raw_toc:
        title = (title or "").strip()

        # Try to capture section number + title (e.g., "2.4.1 Overview")
        m = re.match(r'^(\d+(?:\.\d+)*)\s*(.+)$', title)
        if m:
            section_id = m.group(1)
            sec_title = m.group(2).strip()
        else:
            # Generate synthetic ID
            idx = max(0, level - 1)
            counters[idx] += 1
            for j in range(idx + 1, len(counters)):
                counters[j] = 0
            section_id = ".".join(str(counters[i]) for i in range(level) if counters[i] > 0)
            sec_title = title or f"Section {section_id}"

        parent_id = section_id.rsplit(".", 1)[0] if "." in section_id else None
        full_path = f"{section_id} {sec_title}".strip()

        entries.append({
            "doc_title": doc_title,
            "section_id": section_id,
            "title": sec_title,
            "page": int(page),
            "level": int(level),
            "parent_id": parent_id,
            "full_path": full_path,
            "tags": []
        })
    return entries


# Get document title
doc = fitz.open(pdf_path)
doc_title = doc.metadata.get("title") or os.path.basename(pdf_path)
doc.close()

# Normalize TOC
toc_entries = normalize_toc(raw_toc, doc_title)

# Save to JSONL
toc_path = os.path.join(OUT_DIR, "usb_pd_toc.jsonl")
with open(toc_path, "w", encoding="utf-8") as f:
    for entry in toc_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ TOC saved to {toc_path} with {len(toc_entries)} entries")


✅ TOC saved to /content/output_json/usb_pd_toc.jsonl with 1360 entries


In [6]:
import pdfplumber

def extract_sections(pdf_path, toc_entries):
    specs = []
    with pdfplumber.open(pdf_path) as pdf:
        n_pages = len(pdf.pages)

        for i, entry in enumerate(toc_entries):
            start = max(1, min(entry["page"], n_pages))  # clamp to valid range

            if i + 1 < len(toc_entries):
                next_page = toc_entries[i+1]["page"]
                end = max(start, min(next_page - 1, n_pages))
            else:
                end = n_pages  # last section goes to end

            start_idx, end_idx = start - 1, end - 1
            page_texts = []
            table_count = 0

            for p in range(start_idx, end_idx + 1):
                if 0 <= p < n_pages:  # double safety check
                    page = pdf.pages[p]
                    text = page.extract_text() or ""
                    page_texts.append(text)
                    tables = page.extract_tables()
                    if tables:
                        table_count += len(tables)

            content = "\n".join(page_texts).strip()

            spec_entry = {
                **entry,
                "start_page": start,
                "end_page": end,
                "num_tables": table_count,
                "content": content
            }
            specs.append(spec_entry)

    return specs


spec_entries = extract_sections(pdf_path, toc_entries)

# Save as JSONL
spec_path = os.path.join(OUT_DIR, "usb_pd_spec.jsonl")
with open(spec_path, "w", encoding="utf-8") as f:
    for entry in spec_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ Spec saved to {spec_path} with {len(spec_entries)} sections")


✅ Spec saved to /content/output_json/usb_pd_spec.jsonl with 1360 sections


In [7]:
import fitz
import json
import datetime

doc = fitz.open(pdf_path)
metadata = {
    "doc_title": doc.metadata.get("title") or os.path.basename(pdf_path),
    "file_name": os.path.basename(pdf_path),
    "author": doc.metadata.get("author"),
    "total_pages": doc.page_count,
    "extracted_at_utc": datetime.datetime.utcnow().isoformat() + "Z"
}
doc.close()

meta_path = os.path.join(OUT_DIR, "usb_pd_metadata.jsonl")
with open(meta_path, "w", encoding="utf-8") as f:
    f.write(json.dumps(metadata, ensure_ascii=False) + "\n")

print(f"✅ Metadata saved to {meta_path}")
print(metadata)


✅ Metadata saved to /content/output_json/usb_pd_metadata.jsonl
{'doc_title': 'USB_PD_R3_2 V1.1 2024-10.pdf', 'file_name': 'USB_PD_R3_2 V1.1 2024-10.pdf', 'author': '', 'total_pages': 0, 'extracted_at_utc': '2025-08-13T19:18:31.267691Z'}


In [8]:
import pandas as pd

# Load TOC & spec into DataFrames
toc_df = pd.DataFrame(toc_entries)
spec_df = pd.DataFrame(spec_entries)

# Merge to compare
merged = toc_df.merge(
    spec_df[["section_id", "start_page", "end_page", "num_tables"]],
    on="section_id",
    how="left"
)

# Identify issues
missing = merged[merged["start_page"].isna()]
page_mismatch = merged[
    (~merged["start_page"].isna()) &
    (merged["page"] != merged["start_page"])
]

summary = {
    "total_toc": len(toc_df),
    "total_parsed": len(spec_df),
    "missing_parsed": len(missing),
    "page_mismatches": len(page_mismatch),
    "total_tables": int(spec_df["num_tables"].sum()) if "num_tables" in spec_df else 0
}

# Save report to Excel
val_path = os.path.join(OUT_DIR, "validation_report.xlsx")
with pd.ExcelWriter(val_path, engine="openpyxl") as writer:
    pd.DataFrame([summary]).to_excel(writer, sheet_name="Summary", index=False)
    toc_df.to_excel(writer, sheet_name="TOC", index=False)
    spec_df.to_excel(writer, sheet_name="Parsed", index=False)
    missing.to_excel(writer, sheet_name="MissingParsed", index=False)
    page_mismatch.to_excel(writer, sheet_name="PageMismatches", index=False)

print(f"✅ Validation report saved to {val_path}")
print("Summary:", summary)


✅ Validation report saved to /content/output_json/validation_report.xlsx
Summary: {'total_toc': 1360, 'total_parsed': 1360, 'missing_parsed': 0, 'page_mismatches': 1382, 'total_tables': 0}


In [9]:
import shutil

zip_path = "/content/usb_pd_parsed.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', OUT_DIR)

print(f"✅ All outputs zipped at {zip_path}")


✅ All outputs zipped at /content/usb_pd_parsed.zip


In [11]:
from google.colab import files
files.download("/content/usb_pd_parsed.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>